In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist 
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import mlflow
import optuna

/Users/marcosbautista/uni/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
(x_train,y_train),(x_test,y_test) = mnist.load_data()

#normalizando 
x_train = x_train.astype("float32")/255.0 
x_test = x_test.astype("float32")/255.0 

#vectorizando 
x_train = x_train.reshape(-1,28*28)
x_test = x_test.reshape(-1,28*28)

print(x_train.shape)


(60000, 784)


In [3]:
import os
import mlflow
import dagshub

# Inicializa la integración con DagsHub (te pedirá un token o autenticación por navegador la primera vez)
dagshub.init(repo_owner='bautistammarcos', repo_name='red-densa-mnist', mlflow=True)

# El URI de seguimiento y el experimento se configuran automáticamente
mlflow.set_experiment("Red_Densa_Optuna")

Accessing as bautistammarcos

Initialized MLflow to track repo "bautistammarcos/red-densa-mnist"

Repository bautistammarcos/red-densa-mnist initialized!

<Experiment: artifact_location='mlflow-artifacts:/7a4775be39db41b68d02b7854663cabb', creation_time=1789361687921, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789361687921, lifecycle_stage='active', name='Red_Densa_Optuna', tags={}, trace_location=None, workspace='default'>

In [4]:
mlflow.set_experiment("Red_Densa")


<Experiment: artifact_location='mlflow-artifacts:/8288856dd25b42d2a4face5b84da099c', creation_time=1789361755120, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789361755120, lifecycle_stage='active', name='Red_Densa', tags={}, trace_location=None, workspace='default'>

In [5]:

def objective(trial):

    n_layers = trial.suggest_int("n_layers", 1, 4)

    activation = trial.suggest_categorical(
        "activation",
        ["relu", "tanh", "elu"]
    )

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        ["adam", "rmsprop", "sgd"]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    model = Sequential()

    model.add(Dense(
        trial.suggest_int("units_0", 64, 512, step=64),
        activation=activation,
        input_shape=(784,)
    ))

    for i in range(1, n_layers):
        model.add(Dense(
            trial.suggest_int(
                f"units_{i}",
                64,
                512,
                step=64
            ),
            activation=activation
        ))

    model.add(Dense(10, activation="softmax"))

    optimizer = {
        "adam": tf.keras.optimizers.Adam,
        "rmsprop": tf.keras.optimizers.RMSprop,
        "sgd": tf.keras.optimizers.SGD
    }[optimizer_name](learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Inicia un Run anidado para cada Trial de Optuna
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):

        mlflow.log_params(trial.params)

        history = model.fit(
            x_train,
            y_train,
            validation_split=0.2,
            epochs=10,
            batch_size=128,
            verbose=0
        )

        best_acc = max(history.history["val_accuracy"])

        # Registro de métricas clave del experimento
        mlflow.log_metric("best_val_accuracy", best_acc)
        mlflow.log_metric("final_train_accuracy", history.history["accuracy"][-1])
        mlflow.log_metric("final_val_loss", history.history["val_loss"][-1])

    return best_acc

In [7]:
with mlflow.start_run(run_name="Optuna_Search_Parent"):
    study = optuna.create_study(direction="maximize")

    study.optimize(
        objective,
        n_trials=30
    )

    # Registrar los mejores hiperparámetros globales al finalizar
    mlflow.log_params({f"best_{k}": v for k, v in study.best_trial.params.items()})
    mlflow.log_metric("global_best_val_accuracy", study.best_trial.value)

[I 2026-09-13 22:58:09,691] A new study created in memory with name: no-name-537ab563-2cd2-4127-b489-ae7a7079966a
/Users/marcosbautista/uni/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2026-09-13 22:58:18,794] Trial 0 finished with value: 0.9713333249092102 and parameters: {'n_layers': 2, 'activation': 'tanh', 'optimizer': 'rmsprop', 'learning_rate': 0.00025318374110843935, 'units_0': 384, 'units_1': 512}. Best is trial 0 with value: 0.9713333249092102.


🏃 View run trial_0 at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1/runs/bc5ca0bbbd884dfaa1eff91015cc8359
🧪 View experiment at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1


[I 2026-09-13 22:58:36,468] Trial 1 finished with value: 0.9670833349227905 and parameters: {'n_layers': 2, 'activation': 'tanh', 'optimizer': 'rmsprop', 'learning_rate': 0.00020192253113594948, 'units_0': 448, 'units_1': 64}. Best is trial 0 with value: 0.9713333249092102.


🏃 View run trial_1 at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1/runs/479d958e7fc84cf7a67c185e0c122341
🧪 View experiment at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1


[I 2026-09-13 22:58:42,481] Trial 2 finished with value: 0.9712499976158142 and parameters: {'n_layers': 4, 'activation': 'elu', 'optimizer': 'adam', 'learning_rate': 0.0010188407737410914, 'units_0': 64, 'units_1': 128, 'units_2': 512, 'units_3': 128}. Best is trial 0 with value: 0.9713333249092102.


...

🏃 View run trial_27 at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1/runs/a4fdc8e0843e4f7e800ca329e8ff1707
🧪 View experiment at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1


[I 2026-09-13 23:03:56,444] Trial 28 finished with value: 0.9726666808128357 and parameters: {'n_layers': 1, 'activation': 'relu', 'optimizer': 'rmsprop', 'learning_rate': 0.009180593316996005, 'units_0': 320}. Best is trial 13 with value: 0.9794999957084656.


🏃 View run trial_28 at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1/runs/5b5564c6b1c449e69532537cfc6843b1
🧪 View experiment at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1
🏃 View run trial_29 at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1/runs/82f4ac99e61c458ca742bcbcce78a255
🧪 View experiment at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1


[I 2026-09-13 23:04:03,917] Trial 29 finished with value: 0.9765833616256714 and parameters: {'n_layers': 2, 'activation': 'relu', 'optimizer': 'adam', 'learning_rate': 0.00200887837892421, 'units_0': 448, 'units_1': 192}. Best is trial 13 with value: 0.9794999957084656.


🏃 View run Optuna_Search_Parent at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1/runs/3a9eefd1bde14f7783b50d2adff7b7df
🧪 View experiment at: https://dagshub.com/bautistammarcos/red-densa-mnist.mlflow/#/experiments/1


In [8]:
print(study.best_trial.value)

print(study.best_trial.params)

0.9794999957084656
{'n_layers': 1, 'activation': 'relu', 'optimizer': 'adam', 'learning_rate': 0.0012401052341003264, 'units_0': 512}
